In [5]:
import os
import bs4
from dotenv import load_dotenv
from uuid import uuid4

from langchain_classic.retrievers import MultiQueryRetriever
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_core.stores import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.tracers import LangChainTracer
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

In [6]:
tracer = LangChainTracer()

# Chunking

## 1. Fixed-size / recursive splitting

`chunk_size` and `chunk_overlap` are the only knobs. The splitter walks a separator hierarchy (`"\n\n"` then `"\n"` then `" "` then `""`) so cuts land on natural boundaries where possible, and the overlap repeats boundary text so facts that straddle a cut stay retrievable.

In [12]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
chunk_docs = loader.load()

recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
recursive_chunks = recursive_splitter.split_documents(chunk_docs)

lengths = [len(chunk.page_content) for chunk in recursive_chunks]
print(f"chunks={len(recursive_chunks)} min={min(lengths)} mean={sum(lengths) // len(lengths)} max={max(lengths)}")
print(recursive_chunks[0].page_content[:400])

chunks=66 min=18 mean=699 max=999
LLM Powered Autonomous Agents | Lil'Log






































Lil'Log

















|






Posts




Archive




Search




Tags




FAQ









      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


 


Table of Contents



Agent System Overview

Component One: Planning

Task Decomposition

Self-Reflection


C


In [13]:
for size, overlap in [(200, 0), (500, 50), (1000, 200)]:
    chunks = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap).split_documents(chunk_docs)
    mean = sum(len(chunk.page_content) for chunk in chunks) // len(chunks)
    print(f"size={size:<5} overlap={overlap:<4} -> {len(chunks):>3} chunks, mean {mean} chars")

# Character splitting honours a single separator, so a chunk can grow past chunk_size
character_chunks = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=200).split_documents(chunk_docs)
character_lengths = [len(chunk.page_content) for chunk in character_chunks]
print(f"character -> {len(character_chunks)} chunks, max {max(character_lengths)} chars (chunk_size=1000)")

# Token based: the budget matches the model tokenizer instead of characters
token_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name="gpt-4o", chunk_size=250, chunk_overlap=25
)
token_chunks = token_splitter.split_documents(chunk_docs)
token_mean = sum(len(chunk.page_content) for chunk in token_chunks) // len(token_chunks)
print(f"tiktoken -> {len(token_chunks)} chunks, mean {token_mean} chars per 250-token chunk")

Created a chunk of size 2671, which is longer than the specified 1000
Created a chunk of size 1373, which is longer than the specified 1000
Created a chunk of size 1281, which is longer than the specified 1000
Created a chunk of size 2352, which is longer than the specified 1000
Created a chunk of size 1731, which is longer than the specified 1000
Created a chunk of size 1067, which is longer than the specified 1000
Created a chunk of size 1475, which is longer than the specified 1000
Created a chunk of size 2881, which is longer than the specified 1000
Created a chunk of size 1980, which is longer than the specified 1000
Created a chunk of size 4145, which is longer than the specified 1000
Created a chunk of size 2528, which is longer than the specified 1000


size=200   overlap=0    -> 301 chunks, mean 143 chars
size=500   overlap=50   -> 136 chunks, mean 329 chars
size=1000  overlap=200  ->  66 chunks, mean 699 chars
character -> 41 chunks, max 4144 chars (chunk_size=1000)
tiktoken -> 58 chunks, mean 769 chars per 250-token chunk


## 2. Structure-aware splitting

Splitting on the document's own structure keeps each chunk inside a single section and copies the header path into metadata, which is what metadata filters and routing key off later.

In [14]:
markdown_document = """# Indexing

Chunking decides what a single vector represents.

## Chunking

Recursive splitting walks a separator hierarchy.

## RAPTOR

RAPTOR clusters and summarizes chunks into a tree.

# Retrieval

Retrieval fetches the chunks closest to the query.
"""

header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "h1"), ("##", "h2")])
header_chunks = header_splitter.split_text(markdown_document)

for header_chunk in header_chunks:
    print(header_chunk.metadata, "->", header_chunk.page_content[:60].replace("\n", " "))

{'h1': 'Indexing'} -> Chunking decides what a single vector represents.
{'h1': 'Indexing', 'h2': 'Chunking'} -> Recursive splitting walks a separator hierarchy.
{'h1': 'Indexing', 'h2': 'RAPTOR'} -> RAPTOR clusters and summarizes chunks into a tree.
{'h1': 'Retrieval'} -> Retrieval fetches the chunks closest to the query.


## 3. Semantic (embedding-distance) splitting

Fixed sizes cut through topics. Semantic splitting embeds each sentence one neighbour deep, measures the cosine distance to the next sentence, and cuts where the distance exceeds the 95th percentile of all distances, so boundaries land where the topic actually shifts.

`langchain-experimental`'s `SemanticChunker` implements exactly this and is being sunset, so the algorithm is re-implemented below with its defaults (`breakpoint_threshold_type="percentile"`, amount `95`, `buffer_size=1`), plus a degenerate-chunk merge and a max-size guard the reference implementation lacks: nothing stops a distance-based chunk from growing past the embedding model's window.

In [15]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [16]:
import re

import numpy as np

SENTENCE_SPLIT_REGEX = r"(?<=[.?!])\s+"
BUFFER_SIZE = 1
BREAKPOINT_PERCENTILE = 95
MIN_CHUNK_SIZE = 500
MAX_CHUNK_SIZE = 2000  # bge-small-en-v1.5 truncates input beyond roughly 512 tokens


def semantic_chunks(text, embedding_model):
    sentences = [sentence for sentence in re.split(SENTENCE_SPLIT_REGEX, text) if sentence.strip()]

    # Judge every sentence together with its neighbours, not in isolation
    combined = [
        " ".join(sentences[max(0, index - BUFFER_SIZE) : index + BUFFER_SIZE + 1])
        for index in range(len(sentences))
    ]

    # Embeddings are unit normalised, so cosine distance is 1 - dot product
    vectors = np.array(embedding_model.embed_documents(combined))
    distances = 1 - np.sum(vectors[:-1] * vectors[1:], axis=1)

    threshold = np.percentile(distances, BREAKPOINT_PERCENTILE)
    topic_shifts = set(np.where(distances > threshold)[0].tolist())

    chunks = []
    current = []
    for index, sentence in enumerate(sentences):
        current.append(sentence)
        candidate = " ".join(current)

        # Cut on a topic shift, but only once the chunk carries retrievable content,
        # and always before the embedding model would truncate the chunk
        if (index in topic_shifts and len(candidate) >= MIN_CHUNK_SIZE) or len(candidate) >= MAX_CHUNK_SIZE:
            chunks.append(candidate)
            current = []

    if current:
        chunks.append(" ".join(current))

    return chunks


semantic_chunk_texts = semantic_chunks(chunk_docs[0].page_content, embeddings)
semantic_lengths = [len(chunk) for chunk in semantic_chunk_texts]
print(f"chunks={len(semantic_lengths)} min={min(semantic_lengths)} mean={sum(semantic_lengths) // len(semantic_lengths)} max={max(semantic_lengths)}")
print(semantic_chunk_texts[0][:400])

chunks=27 min=545 mean=1618 max=2222






LLM Powered Autonomous Agents | Lil'Log






































Lil'Log

















|






Posts




Archive




Search




Tags




FAQ









      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


 


Table of Contents



Agent System Overview

Component One: Planning

Task Decomposition

Self-Reflecti


## 4. Agentic (LLM-proposed) splitting

Instead of a rule, ask an LLM where the topic shifts. The model returns sentence indices only, so the text is sliced verbatim from the source and can never be paraphrased or invented.

The number of sentences per call is capped: LumberChunker prompts iteratively over a group of sequential passages rather than the whole document, which also keeps the model from skipping the tool call on long inputs. Cost is one LLM call per window, the highest of the strategies here.

In [22]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field

AGENTIC_WINDOW = 8  # sentences per LLM call
AGENTIC_OVERLAP = 2  # sentences repeated between windows


class ChunkBoundaries(BaseModel):
    """Sentence indices where the content starts a new, self-contained topic."""

    start_indices: list[int] = Field(
        description="0-based indices of the sentences that begin a new chunk. Always include index 0."
    )


agentic_prompt = ChatPromptTemplate.from_template(
    """Split the document into chunks that are each semantically self-contained. You must call the
ChunkBoundaries tool with the 0-based index of every sentence that starts a new chunk; index 0 always
starts the first chunk. Never separate a sentence from the sentence it directly depends on: pronouns,
examples and enumerations belong to the sentence that introduces them.

Numbered sentences:
{sentences}"""
)

agentic_chain = agentic_prompt | ChatGroq(model="openai/gpt-oss-120b", temperature=0).with_structured_output(
    ChunkBoundaries
)

agentic_source = chunk_docs[0].page_content[:3500]
agentic_sentences = [sentence for sentence in re.split(SENTENCE_SPLIT_REGEX, agentic_source) if sentence.strip()]

# Prompt one window at a time; indices stay absolute to the document (start=window_start)
boundary_indices = set()
step = AGENTIC_WINDOW - AGENTIC_OVERLAP
for window_start in range(0, len(agentic_sentences), step):
    sentences = agentic_sentences[window_start : window_start + AGENTIC_WINDOW]
    if len(sentences) < 2:
        break

    numbered_sentences = "\n".join(
        f"{index}: {sentence}" for index, sentence in enumerate(sentences, start=window_start)
    )
    boundaries = agentic_chain.invoke({"sentences": numbered_sentences}, config={"callbacks": [tracer]})
    boundary_indices.update(index for index in boundaries.start_indices if 0 <= index < len(agentic_sentences))
    print(f"window {window_start}-{window_start + len(sentences) - 1} -> {boundaries.start_indices}")

starts = sorted({0, *boundary_indices})
sentence_starts = starts + [len(agentic_sentences)]
agentic_chunk_texts = [
    " ".join(agentic_sentences[sentence_starts[position] : sentence_starts[position + 1]])
    for position in range(len(starts))
]

print(f"sentences={len(agentic_sentences)} chunks={len(agentic_chunk_texts)}")
print([len(chunk) for chunk in agentic_chunk_texts])
print(agentic_chunk_texts[0][300:600])

window 0-7 -> [0, 1, 2, 3, 5, 8, 11]
window 6-13 -> [0, 1, 2, 3, 4, 5, 7]
window 12-19 -> [0, 3, 7]
window 18-23 -> [0, 3]
sentences=24 chunks=9
[737, 104, 155, 293, 199, 344, 262, 176, 1212]
ble of Contents



Agent System Overview

Component One: Planning

Task Decomposition

Self-Reflection


Component Two: Memory

Types of Memory

Maximum Inner Product Search (MIPS)


Component Three: Tool Use

Case Studies

Scientific Discovery Agent

Generative Agents Simulation

Proof-of-Concept E


## 5. Comparison

| Strategy | Mechanism | Cost | Best for |
|---|---|---|---|
| Recursive / fixed | Separator hierarchy, `chunk_size` + `chunk_overlap` | Free, local | Default for unstructured prose |
| Token-based | Split on the model tokenizer | Free, local | Matching an embedding or prompt token budget |
| Structure-aware | Split on headers, header path becomes metadata | Free, local | Docs, wikis and code with headings |
| Semantic | Embed sentences, cut above the 95th percentile distance | 1 embedding call per sentence | Topic shifts without structural markers |
| Agentic | LLM returns the indices where the topic shifts | 1 LLM call per passage window | High-value long-form narrative |

Overlap is boundary insurance, not a free win: in Chroma's token-level evaluation, Recursive-400 with no overlap scored 89.9 recall at 5 retrieved chunks against 88.3 with a 200-token overlap, while at minimum retrieval the overlap variant scored 73.3 against 61.4 `(Smith & Troynikov, 2024)`.

# Multi-representation Indexing

In [7]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()

loader = WebBaseLoader("https://lilianweng.github.io/posts/2024-02-05-human-data-quality/")
docs.extend(loader.load())

In [8]:
print(len(docs))
print(docs[0].page_content[500:1000])


2
Tool Use

Case Studies

Scientific Discovery Agent

Generative Agents Simulation

Proof-of-Concept Examples


Challenges

Citation

References





Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
A


In [9]:
chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{doc}")
    | ChatGroq(model="openai/gpt-oss-20b", temperature=0)
    | StrOutputParser()
)

for doc in docs:
    doc.page_content = doc.page_content[:4000]

summaries = [chain.invoke(doc, config={"callbacks": [tracer]}) for doc in docs]

print(summaries)

['**LLM‑Powered Autonomous Agents – Summary**\n\nLilian\u202fWeng’s June\u202f2023 post outlines how large language models (LLMs) can serve as the “brain” of autonomous agents, complementing three core modules: **planning**, **memory**, and **tool use**.\n\n1. **Planning** – The agent breaks a complex task into sub‑goals and refines its actions through self‑reflection. Techniques include:\n   * **Chain‑of‑Thought (CoT)** prompting, which forces the model to reason step‑by‑step.\n   * **Tree‑of‑Thoughts** (Yao\u202fet\u202fal.,\u202f2023), which explores multiple reasoning paths in a tree structure, evaluated by a classifier or majority vote.\n   * **LLM+P** (Liu\u202fet\u202fal.,\u202f2023), where the LLM translates a problem into PDDL, a classical planner generates a plan, and the plan is translated back into natural language.\n\n2. **Memory** – Two layers:\n   * **Short‑term memory** via in‑context learning (prompt engineering).\n   * **Long‑term memory** using an external vector sto

In [10]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

id_key = "doc_id"
doc_ids = [str(uuid4()) for _ in docs]

summary_docs = [
    Document(page_content=summary, metadata={id_key: doc_id})
    for summary, doc_id in zip(summaries, doc_ids)
]

vectorstore = Chroma(
    collection_name="summaries",
    embedding_function=embeddings,
    persist_directory="chroma",
)
vectorstore.add_documents(summary_docs, ids=doc_ids)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

['bd04c24b-a453-4799-ba40-5f998394dab2',
 'f7001842-a7f4-42a0-9fca-9cf76b781eac']

In [11]:
from langchain_classic.retrievers import MultiVectorRetriever

store = InMemoryStore()
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key,
    search_kwargs={"k": 4},
)

store.mset(list(zip(doc_ids, docs)))

retrieved_docs = retriever.invoke("How do agents use tools?")
print(len(retrieved_docs))
print(retrieved_docs[0].page_content[:500])

2






LLM Powered Autonomous Agents | Lil'Log






































Lil'Log

















|






Posts




Archive




Search




Tags




FAQ









      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


 


Table of Contents



Agent System Overview

Component One: Planning

Task Decomposition

Self-Reflection


Component Two: Memory

Types of Memory

Maximum Inner Product Search (MIPS)


Component Three: 


# RAPTOR

Level 0 is the raw chunks. Every level above clusters the embeddings of the level below, summarizes each cluster with the LLM and then recurses, so retrieval can return an abstraction instead of a fragment.

The paper reduces dimensions with UMAP before fitting a Gaussian mixture; `umap-learn` is not a dependency here, so the mixture is fitted on the raw 384-dimensional embeddings. Each summarization call is truncated to a bounded character budget, because the LLM only ever sees a cluster of chunks, never the whole document.

In [ ]:
import numpy as np
from sklearn.mixture import GaussianMixture

raptor_loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
raptor_docs = raptor_loader.load()

raptor_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
raptor_chunks = raptor_splitter.split_documents(raptor_docs)

print(f"level 0: {len(raptor_chunks)} chunks")

In [ ]:
N_CLUSTERS = 4
CLUSTER_PROB_THRESHOLD = 0.1  # soft clustering as in the paper: a node may join several clusters
MAX_CLUSTER_CHARS = 4000  # bound the summarization prompt, never the whole document
MAX_LEVELS = 2

summarize_prompt = ChatPromptTemplate.from_template(
    """Summarize the excerpts below into one concise, factual passage.

Preserve the key entities, methods and numbers. Do not add information that is not present.

Excerpts:
{context}"""
)
summarize_chain = summarize_prompt | ChatGroq(model="openai/gpt-oss-20b", temperature=0) | StrOutputParser()


def build_raptor_tree(chunks, max_levels=MAX_LEVELS):
    levels = {0: chunks}
    nodes = list(chunks)

    for level in range(1, max_levels + 1):
        if len(nodes) < 4:
            break

        texts = [node.page_content for node in nodes]
        vectors = np.array(embeddings.embed_documents(texts))

        # Never ask for more clusters than there are nodes to group
        n_clusters = min(N_CLUSTERS, max(2, len(nodes) // 3))
        mixture = GaussianMixture(n_components=n_clusters, random_state=0).fit(vectors)
        probabilities = mixture.predict_proba(vectors)

        summaries = []
        for cluster in range(n_clusters):
            members = [
                index
                for index, row in enumerate(probabilities)
                if row.argmax() == cluster or row[cluster] > CLUSTER_PROB_THRESHOLD
            ]
            if not members:
                continue

            context = "\n\n".join(texts[index] for index in members)[:MAX_CLUSTER_CHARS]
            summary = summarize_chain.invoke({"context": context}, config={"callbacks": [tracer]})
            summaries.append(
                Document(page_content=summary, metadata={"level": level, "num_children": len(members)})
            )

        print(f"level {level}: {len(summaries)} summaries from {len(nodes)} nodes")
        levels[level] = summaries
        nodes = summaries

    return levels


raptor_levels = build_raptor_tree(raptor_chunks)
raptor_nodes = [node for level_nodes in raptor_levels.values() for node in level_nodes]
print(f"total nodes: {len(raptor_nodes)}")

## Collapsed tree retrieval

Every node, from raw chunk to top-level summary, lives in a single index, so one similarity search can return either a detail or an abstraction. The alternative, tree traversal, searches the root first and descends into the best children level by level.

In [ ]:
raptor_store = Chroma(
    collection_name="raptor",
    embedding_function=embeddings,
    persist_directory="chroma",
)

raptor_ids = [f"level-{node.metadata['level']}-{index}" for index, node in enumerate(raptor_nodes)]
raptor_store.add_documents(raptor_nodes, ids=raptor_ids)

print({level: len(level_nodes) for level, level_nodes in raptor_levels.items()})

In [ ]:
raptor_retriever = raptor_store.as_retriever(search_kwargs={"k": 4})

for retrieved in raptor_retriever.invoke("How does an agent decide which tool to call?"):
    print(f"--- level {retrieved.metadata['level']} ---")
    print(retrieved.page_content[:300])
    print()